# Job Matching Engine
### Pipeline multi-fonte con scoring semantico e LLM

Sistema di **retrieval e ranking di annunci di lavoro**: aggrega offerte da cinque job board tramite API ufficiali, le normalizza in uno schema comune, applica filtri di eleggibilità configurabili e le ordina per affinità con un profilo candidato, combinando **embedding semantici multilingue** e una valutazione **LLM-as-judge**.

Nessuno scraping: solo API pubbliche e documentate.

**La pipeline non è legata a un profilo o a un paese.** Ruoli, skill, aree geografiche, soglie di eleggibilità e testo del candidato sono tutti parametri concentrati in un'unica cella di configurazione: per adattarla a un'altra ricerca non serve toccare il codice. La configurazione fornita — analista dati junior in Piemonte — è un esempio funzionante, non un vincolo.

---

## Architettura

```
                    ┌─────────────────────────────────────────┐
   API sorgenti     │  Adzuna · JSearch · Jooble · Careerjet   │
                    │  · Arbeitnow                            │
                    └──────────────────┬──────────────────────┘
                                       │  normalizzazione → schema comune
                                       ▼
   Deduplica        ┌─────────────────────────────────────────┐
                    │  hash MD5 su (link, titolo, azienda)     │
                    │  + match su URL canonico e titolo|azienda│
                    └──────────────────┬──────────────────────┘
                                       ▼
   Filtri           ┌─────────────────────────────────────────┐
   di eleggibilità  │  categorie protette · seniority /        │
                    │  anni esperienza (regex) · modalità di   │
                    │  lavoro e area geografica                │
                    └──────────────────┬──────────────────────┘
                                       ▼
   Scoring          ┌─────────────────────────────────────────┐
   ibrido           │  cosine similarity su embedding          │
                    │  multilingue (0.8) + keyword match (0.2) │
                    └──────────────────┬──────────────────────┘
                                       ▼
   Re-ranking       ┌─────────────────────────────────────────┐
   LLM              │  Claude valuta i top-N e restituisce     │
                    │  punteggio + motivazione + gap (JSON)    │
                    └──────────────────┬──────────────────────┘
                                       ▼
   Output           ┌─────────────────────────────────────────┐
                    │  CSV locale · Google Sheets (opzionale)  │
                    └─────────────────────────────────────────┘
```

## Scelte tecniche

| Componente | Scelta | Motivazione |
|---|---|---|
| Embedding | `paraphrase-multilingual-MiniLM-L12-v2` | Gli annunci mescolano italiano e inglese, spesso nella stessa descrizione: serve uno spazio vettoriale condiviso fra le due lingue. Modello leggero (~470 MB), gira su CPU. |
| Scoring | ibrido semantico + keyword | Il solo semantico premia annunci genericamente "data" ma privi delle tecnologie richieste; il solo keyword è fragile ai sinonimi. La combinazione 80/20 corregge entrambi i difetti. |
| Re-ranking | LLM su top-N | Chiamare il modello su tutti gli annunci sarebbe costoso e lento. Il ranking semantico fa da retrieval stage, l'LLM da precision stage sui candidati migliori. |
| Deduplica | doppia chiave | Lo stesso annuncio compare su più board con URL diversi: serve sia il match su URL canonico sia quello su (titolo, azienda). |
| Resilienza | try/except per fonte + backoff | Se una board va in errore o rate limit, le altre proseguono. Retry con backoff esponenziale e jitter sui codici transitori (429/500/502/503/529). |

## Requisiti

Python 3.9+. Le dipendenze sono nella Cella 1. Il notebook gira sia su **Google Colab** sia in **locale**: l'export su Google Sheets è opzionale, il fallback è un CSV.

## Come ottenere le chiavi API

Il notebook funziona anche **senza nessuna chiave a pagamento**: Arbeitnow non richiede autenticazione e Adzuna è gratuita. Le altre fonti sono opzionali.

| Fonte | Costo | Cosa serve |
|---|---|---|
| **Arbeitnow** | gratis, nessuna registrazione | niente |
| **Adzuna** | gratis (250 chiamate/giorno) | `ADZUNA_APP_ID` + `ADZUNA_APP_KEY` |
| **Careerjet** | gratis, illimitato | `CAREERJET_AFFID` |
| **Jooble** | gratis (500 richieste/mese) | `JOOBLE_KEY` |
| **JSearch** | freemium (200 richieste/mese nel piano Basic) | `RAPIDAPI_KEY` |
| **Anthropic** | a consumo (~0.01–0.05 € per run con Haiku) | `ANTHROPIC_API_KEY` |

---

### 1. Adzuna — consigliata, è la fonte principale
1. Vai su **https://developer.adzuna.com** e clicca *Sign up*.
2. Conferma l'email e accedi alla dashboard.
3. Trovi subito **Application ID** (8 caratteri) e **Application Key** (32 caratteri esadecimali).
4. Nessuna carta richiesta. Limite: 250 chiamate al giorno, più che sufficienti.

### 2. Careerjet — gratuita e senza limiti
1. Vai su **https://www.careerjet.com/partners/api/** e registrati come affiliato.
2. Ricevi un **Affiliate ID** di 20 caratteri: è il `CAREERJET_AFFID`.
3. L'API richiede i parametri `user_ip` e `user_agent` per il tracking degli affiliati. In un contesto server-side non c'è un IP utente reale: nel notebook sono valorizzati con placeholder, come previsto dalla documentazione per le integrazioni non-browser.

### 3. Jooble
1. Vai su **https://jooble.org/api/about** e compila il form di richiesta.
2. Ricevi via email una chiave in formato UUID (es. `a1b2c3d4-0000-0000-0000-000000000000`): è il `JOOBLE_KEY`.
3. Attenzione: la chiave va nell'**URL**, non negli header — è il comportamento previsto dalla loro API.

### 4. JSearch (via RapidAPI) — la più ricca ma con quota stretta
1. Crea un account su **https://rapidapi.com**.
2. Cerca **JSearch** e clicca *Subscribe to Test*.
3. Scegli il piano **Basic** (gratuito, 200 richieste/mese). Richiede una carta per la verifica ma non addebita nulla sul piano free.
4. Nella scheda *Endpoints* trovi la chiave `X-RapidAPI-Key`: è il `RAPIDAPI_KEY`.
5. Con 200 richieste/mese conviene tenerla spenta nelle run quotidiane e accenderla solo quando serve una ricerca più ampia.

### 5. Anthropic — per il re-ranking LLM
1. Vai su **https://console.anthropic.com** e crea un account.
2. Sezione **Billing**: carica un credito iniziale (bastano 5 $, durano mesi con questo carico di lavoro).
3. Sezione **API Keys** → *Create Key*. La chiave inizia con `sk-ant-`.
4. **La chiave viene mostrata una sola volta**: copiala subito. Se la perdi, revocala e creane un'altra.
5. Il notebook usa un modello economico e valuta solo i primi N annunci, quindi il costo per run resta nell'ordine dei centesimi.

---

### Dove mettere le chiavi

**Mai nel codice.** Il notebook le legge, in ordine di priorità:

1. **Colab Secrets** — icona 🔑 nella barra laterale sinistra di Colab. Aggiungi ogni chiave con il nome esatto della variabile e attiva *Notebook access*. È il metodo consigliato: le chiavi restano nel tuo account Google, non nel file `.ipynb`.
2. **Variabili d'ambiente** — in locale, crea un file `.env` (già in `.gitignore`) partendo da `.env.example`.
3. **Input manuale** — se non trova nulla, il notebook chiede la chiave con `getpass`, che non la stampa a schermo né la salva negli output.

In [ ]:
# ============================================================================
# CELLA 1 — Dipendenze
# ============================================================================
!pip install -q sentence-transformers requests pandas anthropic gspread gspread-dataframe python-dotenv
print("Dipendenze installate.")

In [ ]:
# ============================================================================
# CELLA 2 — Caricamento credenziali
# ----------------------------------------------------------------------------
# Nessuna chiave e scritta in questo file. Ordine di ricerca:
#   1) Colab Secrets  2) variabili d'ambiente / file .env  3) input manuale
# ============================================================================
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()                      # carica .env se presente (esecuzione locale)
except Exception:
    pass                               # dotenv assente o non applicabile: si prosegue

try:
    from google.colab import userdata  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def get_secret(nome, obbligatoria=False):
    """Recupera una credenziale dalla prima fonte disponibile. None se assente."""
    if IN_COLAB:
        try:
            valore = userdata.get(nome)
            if valore:
                return valore.strip()
        except Exception:
            pass                       # secret non definito o accesso non concesso

    valore = os.environ.get(nome)
    if valore:
        return valore.strip()

    if obbligatoria:
        return getpass(f"Inserisci {nome}: ").strip()
    return None


ADZUNA_APP_ID     = get_secret("ADZUNA_APP_ID")
ADZUNA_APP_KEY    = get_secret("ADZUNA_APP_KEY")
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
RAPIDAPI_KEY      = get_secret("RAPIDAPI_KEY")
JOOBLE_KEY        = get_secret("JOOBLE_KEY")
CAREERJET_AFFID   = get_secret("CAREERJET_AFFID")

# Le fonti si accendono da sole se la chiave corrispondente e disponibile.
USA_ARBEITNOW = True                                    # nessuna chiave richiesta
USA_ADZUNA    = bool(ADZUNA_APP_ID and ADZUNA_APP_KEY)
USA_JSEARCH   = bool(RAPIDAPI_KEY)
USA_JOOBLE    = bool(JOOBLE_KEY)
USA_CAREERJET = bool(CAREERJET_AFFID)
USA_LLM       = bool(ANTHROPIC_API_KEY)

_stato = {
    "Arbeitnow": USA_ARBEITNOW, "Adzuna": USA_ADZUNA, "JSearch": USA_JSEARCH,
    "Jooble": USA_JOOBLE, "Careerjet": USA_CAREERJET, "Scoring LLM": USA_LLM,
}
print("Ambiente:", "Google Colab" if IN_COLAB else "locale")
for nome, attiva in _stato.items():
    print(f"  {'ON ' if attiva else 'off'}  {nome}")
if not any([USA_ADZUNA, USA_JSEARCH, USA_JOOBLE, USA_CAREERJET, USA_ARBEITNOW]):
    raise SystemExit("Nessuna fonte attiva: configura almeno una chiave (vedi cella precedente).")

## Adattare la pipeline alla propria ricerca

Tutto ciò che riguarda **chi cerca**, **cosa cerca** e **dove** è concentrato nella cella di configurazione che segue. Nessun'altra cella va toccata: il resto del notebook legge da lì.

I sei blocchi da personalizzare:

| § | Cosa definisce | Esempio di modifica |
|---|---|---|
| **1** | Il profilo del candidato | Incolla il testo del tuo CV al posto di quello di esempio |
| **2** | I ruoli da cercare | Sostituisci i blocchi con le tue mansioni; puoi averne 2, 3 o 10 |
| **3** | Le skill che valgono un bonus | Elenca le tecnologie del tuo settore |
| **4** | Paese, lingua e area geografica | Cambia `PAESE`, `LOCALE` e definisci le tue aree con le relative città |
| **5** | I filtri di eleggibilità | Alza o abbassa la soglia di esperienza, disattiva i filtri che non ti servono |
| **6** | I parametri tecnici | Pagine per fonte, freschezza degli annunci, pesi dello scoring |

### Tre esempi di configurazione

**Data analyst junior a Torino** (configurazione di esempio già presente)
```python
PAESE, LOCALE = "it", "it_IT"
LUOGHI = ["Torino"]
AREE_AMMESSE = ["Piemonte"]
MAX_ANNI_ESPERIENZA = 1
```

**Frontend developer mid-level a Berlino, aperto al remoto**
```python
PAESE, LOCALE = "de", "de_DE"
LUOGHI = ["Berlin"]
AREE_GEOGRAFICHE = {"Berlin-Brandenburg": ["berlin", "potsdam", "brandenburg"]}
AREE_AMMESSE = ["Berlin-Brandenburg"]
MAX_ANNI_ESPERIENZA = 5
FILTRO_CATEGORIE_PROTETTE = False      # normativa specifica italiana
BLOCCHI = {"Frontend": ["frontend developer", "react developer", "ui engineer"]}
```

**Ricerca solo remota, nessun vincolo geografico**
```python
LUOGHI = [""]              # nessun centro di ricerca
AREE_AMMESSE = []          # disattiva il filtro geografico
SOLO_REMOTO = True         # tiene esclusivamente gli annunci da remoto
```

### Un avvertimento sulla ritaratura

Cambiando profilo, la banda di riscalatura dei punteggi (`SIM_MIN` / `SIM_MAX`, § 6) resta calibrata sul profilo precedente. Dopo la prima run guarda la distribuzione della colonna `score_semantico`: se i punteggi sono tutti schiacciati verso 0 o verso 100, sposta i due estremi. La cella di scoring stampa media e massimo proprio per rendere immediato questo controllo.

In [ ]:
# ============================================================================
# CELLA 3 — CONFIGURAZIONE
# ----------------------------------------------------------------------------
# L'UNICA cella da modificare per adattare la pipeline a un'altra ricerca.
# I valori presenti sono un esempio funzionante (analista dati junior a Torino):
# sostituiscili con i propri. Ogni sezione e indipendente dalle altre.
# ============================================================================
from datetime import date

# ============================================================================
# § 1 — PROFILO DEL CANDIDATO
# ----------------------------------------------------------------------------
# Testo di riferimento per il matching semantico e per il prompt di valutazione.
# Incolla qui il tuo CV in forma discorsiva: piu e specifico su tecnologie e
# ambiti, piu il ranking sara preciso. Elenchi puntati e date non servono, il
# modello lavora sul contenuto semantico. Indicativamente 150-300 parole.
# ============================================================================
PROFILO_CANDIDATO = """
Laureato magistrale in Metodi Statistici ed Economici per le Decisioni (Statistica), Universita di Torino.
Sviluppo di modelli statistici e di machine learning su dati reali, in particolare serie storiche economico-finanziarie.
Padronanza dell'intero flusso analitico: preparazione e pulizia dati, modellizzazione, validazione, interpretazione.
Linguaggi: Python (pandas, NumPy, scikit-learn, TensorFlow, Keras), R, SQL, T-SQL, STATA.
Machine learning e AI: modelli supervisionati e non supervisionati, Random Forest, boosting, BART, reti neurali LSTM,
modelli probabilistici HMM, reti bayesiane DAG, deep learning. Uso di strumenti di AI generativa e costruzione di agenti AI.
Statistica ed econometria: analisi delle serie storiche, analisi multivariata, econometria, modelli logit/probit ordinali,
indagini campionarie, simulazione Monte Carlo. Business Intelligence: Power BI (DAX, dashboard), Excel avanzato,
Crystal Reports, business analysis, SQL Server, ERP Panthera. Validazione walk-forward, ensemble, test statistici.
Tesi magistrale: modello ibrido HMM-LSTM per la previsione probabilistica dei rendimenti dell'S&P 500,
pipeline end-to-end in Python, validazione walk-forward, ensemble multi-seed, test statistici formali.
Altri progetti: machine learning per la diagnosi dell'Alzheimer; modello logit ordinale sul benessere delle famiglie
italiane (ISTAT); campionamento adattivo sugli incidenti stradali. Certificazioni: Google AI Essentials, Academy Randstad Digital.
Cerca un primo ruolo (junior, 0-1 anni di esperienza) in data science, data analysis, business analysis o AI/machine learning,
base Torino o remoto. Inglese B2.
"""

# ============================================================================
# § 2 — RUOLI DA CERCARE
# ----------------------------------------------------------------------------
# I ruoli sono divisi in blocchi tematici e ogni run ne esegue UNO SOLO, scelto
# in base al giorno dell'anno. Con N blocchi si copre l'intero set in N giorni,
# tenendo basso il numero di chiamate API per singola esecuzione.
#
# Puoi definire quanti blocchi vuoi: la rotazione si adatta automaticamente.
# Un blocco unico e perfettamente valido se le quote lo permettono.
# Le stringhe vanno scritte come le userebbe un motore di ricerca, non come
# titoli formali: mescolare italiano e inglese aumenta la copertura.
# Per forzare un blocco specifico: FORZA_BLOCCO = 1, 2, 3, ... (None = automatico)
# ============================================================================
FORZA_BLOCCO = None

BLOCCHI = {
    "Blocco 1 - Core data, BI e visualizzazione": [
        "data scientist junior", "data analyst", "junior data analyst", "business analyst",
        "business intelligence analyst", "statistician", "machine learning engineer junior",
        "data engineer junior", "power bi developer", "bi developer", "dashboard developer",
        "data visualization specialist", "sql analyst", "analytics engineer",
        "reporting analyst", "insights analyst", "experimentation analyst",
        "analista dati", "analista di business", "analista bi", "statistico",
    ],
    "Blocco 2 - Finanza, rischio, quant, ricerca e sanita": [
        "quantitative analyst junior", "quantitative researcher junior", "risk analyst",
        "credit risk analyst", "market risk analyst", "model validation analyst",
        "financial analyst", "pricing analyst", "actuarial analyst", "research analyst",
        "fraud analyst", "aml analyst", "biostatistician", "clinical data analyst",
        "healthcare data analyst", "survey statistician",
        "analista di rischio", "analista finanziario", "statistico sanitario", "funzionario statistico",
    ],
    "Blocco 3 - Funzioni aziendali, AI/GenAI, consulenza ed ERP": [
        "marketing analyst", "crm analyst", "digital analyst", "customer insights analyst",
        "product analyst", "growth analyst", "demand planner", "supply chain analyst",
        "operations analyst", "people analytics", "ai analyst", "ai engineer junior",
        "prompt engineer", "nlp analyst", "data analytics consultant junior",
        "business intelligence consultant", "erp analyst",
        "analista marketing", "analista funzionale", "consulente business intelligence",
    ],
}

# ============================================================================
# § 3 — SKILL CHIAVE (bonus keyword)
# ----------------------------------------------------------------------------
# Termini che, se presenti nell'annuncio, alzano il punteggio. Scrivili come
# compaiono nel testo degli annunci, non come li scriveresti sul CV.
# Il contributo satura dopo SATURAZIONE_KEYWORD match (§ 6), quindi un annuncio
# infarcito di buzzword non ottiene un vantaggio sproporzionato.
# ============================================================================
SKILL_CHIAVE = [
    "python", "sql", "t-sql", "power bi", "excel", "r ", "machine learning", "deep learning",
    "scikit-learn", "tensorflow", "keras", "pandas", "numpy", "lstm", "time series",
    "serie storiche", "statistica", "econometria", "data mining", "etl", "git",
    "random forest", "boosting", "business intelligence", "dashboard", "forecasting",
    "previsione", "modelli predittivi", "data analysis", "analisi dati", "gen ai",
    "generative ai", "ai generativa", "llm",
]

# ============================================================================
# § 4 — AREA GEOGRAFICA
# ----------------------------------------------------------------------------
# PAESE : codice ISO a due lettere usato da Adzuna e JSearch.
#         Adzuna supporta: at, au, be, br, ca, ch, de, es, fr, gb, in, it, mx,
#         nl, nz, pl, pt, se, sg, us, za
# LOCALE: codice lingua_PAESE usato da Careerjet (es. "it_IT", "de_DE", "en_GB")
# LUOGHI: centri attorno a cui cercare. [""] = nessun vincolo lato API.
#
# AREE_GEOGRAFICHE definisce le zone riconosciute: a ogni area corrisponde la
# lista di citta che la identificano. Il matching e per sottostringa sul campo
# "luogo" dell'annuncio, ripulito dagli accenti. Servono i nomi delle citta
# perche gli annunci quasi mai indicano la regione: scrivono "Rivoli", non
# "Piemonte". Aggiungi le tue aree con i rispettivi centri principali.
#
# AREE_AMMESSE : quali di quelle aree accettare. Lista vuota = nessun filtro
#                geografico (utile per ricerche solo remote o internazionali).
# SOLO_REMOTO  : se True, tiene esclusivamente gli annunci classificati "Remoto".
# ============================================================================
PAESE  = "it"
LOCALE = "it_IT"

LUOGHI      = ["Torino"]
DISTANZA_KM = 120

AREE_GEOGRAFICHE = {
    "Piemonte": ["piemonte", "torino", "novara", "alessandria", "asti", "cuneo", "biella",
                 "vercelli", "verbania", "verbano", "ivrea", "moncalieri", "rivoli", "collegno",
                 "nichelino", "settimo torinese", "chieri", "pinerolo", "carmagnola", "bra",
                 "alba", "fossano", "casale monferrato", "tortona", "domodossola", "grugliasco",
                 "venaria"],
    "Liguria": ["liguria", "genova", "la spezia", "savona", "imperia", "sanremo", "san remo",
                "chiavari", "rapallo", "sarzana", "albenga", "ventimiglia", "sestri levante",
                "arenzano", "lavagna", "loano", "finale ligure"],
    "Lombardia": ["lombardia", "milano", "bergamo", "brescia", "monza", "como", "varese", "pavia",
                  "cremona", "mantova", "lecco", "lodi", "sondrio", "sesto san giovanni",
                  "cinisello balsamo", "legnano", "busto arsizio", "gallarate", "vigevano", "rho",
                  "cologno monzese", "seregno", "vimercate", "saronno", "cantu", "desio", "crema",
                  "mariano comense", "assago", "segrate", "san donato milanese", "rozzano",
                  "cernusco"],
}

AREE_AMMESSE = ["Piemonte"]
SOLO_REMOTO  = False

# ============================================================================
# § 5 — FILTRI DI ELEGGIBILITA
# ----------------------------------------------------------------------------
# Ogni filtro puo essere disattivato singolarmente. Gli annunci scartati non
# spariscono: finiscono nel foglio "Esclusi" con il motivo, cosi le regole
# restano verificabili.
# ============================================================================

# --- Seniority ed esperienza richiesta ---
# MAX_ANNI_ESPERIENZA = None disattiva del tutto il filtro.
MAX_ANNI_ESPERIENZA = 1

# Marcatori nel TITOLO che indicano un ruolo troppo avanzato.
# Chi cerca un ruolo senior deve svuotare questa lista o invertirne la logica.
MARCATORI_SENIORITY = [
    "senior", "sr.", "sr ", "lead ", " lead", "principal", "staff ",
    "head of", "responsabile", "manager", "team leader", "teamleader",
    "expert", "specialist senior", "architect", "capo ", "direttore",
    "level 3", "livello 3",
]

# Formule che implicano esperienza pluriennale senza indicare un numero.
FORMULE_PLURIENNALI = [
    "esperienza pluriennale", "pluriennale esperienza", "consolidata esperienza",
    "esperienza consolidata", "comprovata esperienza", "solida esperienza",
    "significativa esperienza", "esperienza significativa",
    "extensive experience", "proven experience", "solid experience",
]

# --- Categorie protette (normativa italiana L. 68/99) ---
# Annunci legalmente riservati a chi rientra nelle liste del collocamento mirato.
# FILTRO_CATEGORIE_PROTETTE = False fuori dall'Italia, o se rientri nelle categorie.
# MODALITA_FILTRO: "mirata" richiede riferimento normativo E termine di riserva
#                  (evita i falsi positivi da disclaimer di rito);
#                  "aggressiva" scarta alla prima menzione della normativa.
FILTRO_CATEGORIE_PROTETTE = True
MODALITA_FILTRO = "mirata"

# ============================================================================
# § 6 — PARAMETRI TECNICI
# ----------------------------------------------------------------------------
# Raramente serve toccarli. Le eccezioni sono MAX_GIORNI, se la ricerca rende
# pochi risultati, e SIM_MIN/SIM_MAX, da ritarare quando si cambia profilo.
# ============================================================================
MAX_GIORNI   = 30     # eta massima dell'annuncio, in giorni
PER_PAGINA   = 50
PAGINE       = 2      # pagine per (query, luogo) su Adzuna
PAGINE_ALTRE = 1      # pagine sulle altre fonti: alzare consuma le quote gratuite

PESO_SEMANTICO = 0.8  # i due pesi devono sommare a 1
PESO_KEYWORD   = 0.2
SATURAZIONE_KEYWORD = 8   # oltre questo numero di match il bonus non cresce piu

# Banda di riscalatura della similarita coseno in punteggio 0-100.
# Calibrata sul profilo di esempio: ritarare osservando `score_semantico`.
SIM_MIN, SIM_MAX = 0.10, 0.55

# ============================================================================
# Selezione del blocco e controlli di coerenza — non modificare
# ============================================================================
assert BLOCCHI, "Definisci almeno un blocco di ruoli in § 2."
assert abs(PESO_SEMANTICO + PESO_KEYWORD - 1.0) < 1e-9, "I pesi devono sommare a 1 (§ 6)."
assert all(a in AREE_GEOGRAFICHE for a in AREE_AMMESSE), \
    "Ogni voce di AREE_AMMESSE deve esistere in AREE_GEOGRAFICHE (§ 4)."

_nomi = list(BLOCCHI.keys())
if FORZA_BLOCCO is not None and 1 <= FORZA_BLOCCO <= len(_nomi):
    _indice = FORZA_BLOCCO - 1
else:
    _indice = date.today().timetuple().tm_yday % len(_nomi)   # rotazione su N blocchi
BLOCCO_OGGI = _nomi[_indice]
QUERIES = BLOCCHI[BLOCCO_OGGI]

print(f"Blocco in esecuzione : {BLOCCO_OGGI}")
print(f"Query                : {len(QUERIES)} (blocco {_indice + 1} di {len(_nomi)})")
print(f"Paese / locale       : {PAESE} / {LOCALE}")
print(f"Luoghi               : {LUOGHI or 'nessun vincolo'}")
if SOLO_REMOTO:
    print("Filtro geografico    : solo annunci da remoto")
elif AREE_AMMESSE:
    print(f"Filtro geografico    : {', '.join(AREE_AMMESSE)} (il remoto e sempre ammesso)")
else:
    print("Filtro geografico    : disattivato")
print(f"Esperienza max       : {MAX_ANNI_ESPERIENZA if MAX_ANNI_ESPERIENZA is not None else 'nessun limite'}")
print("Rotazione automatica attiva." if FORZA_BLOCCO is None else f"Blocco forzato ({FORZA_BLOCCO}).")

In [ ]:
# ============================================================================
# CELLA 4 — Raccolta multi-fonte
# ----------------------------------------------------------------------------
# Ogni API ha uno schema di risposta diverso: le funzioni fetch_* si occupano
# della normalizzazione verso il record comune definito da `riga`.
# Ogni fonte e isolata: un errore su una non interrompe le altre.
# ============================================================================
import hashlib
import html as _html
import re
import time

import pandas as pd
import requests


def pulisci_html(testo):
    """Rimuove i tag e normalizza gli spazi: le descrizioni arrivano in HTML grezzo."""
    if not testo:
        return ""
    testo = re.sub(r"<[^>]+>", " ", str(testo))
    testo = _html.unescape(testo)
    return re.sub(r"\s+", " ", testo).strip()


def crea_id(fonte, link, titolo, azienda):
    """ID stabile per annuncio: hash del contenuto identificativo, non della posizione."""
    base = (link or "") + (titolo or "") + (azienda or "")
    return f"{fonte}:" + hashlib.md5(base.encode("utf-8")).hexdigest()[:12]


def riga(fonte, titolo, azienda, luogo, descrizione, link,
         pubblicato="", categoria="", smin=None, smax=None, query=""):
    """Schema comune verso cui converge ogni fonte."""
    return {
        "id": crea_id(fonte, link, titolo, azienda),
        "titolo": (titolo or "").strip(),
        "azienda": (azienda or "").strip(),
        "luogo": (luogo or "").strip(),
        "categoria": categoria or "",
        "descrizione": pulisci_html(descrizione),
        "salario_min": smin,
        "salario_max": smax,
        "pubblicato": (pubblicato or "")[:10],
        "link": link or "",
        "query": query,
        "fonte": fonte,
    }


def get_json(url, **kwargs):
    risposta = requests.get(url, timeout=30, **kwargs)
    risposta.raise_for_status()
    return risposta.json()


def giorni_to_jsearch(giorni):
    """JSearch accetta finestre temporali categoriali, non un numero di giorni."""
    if giorni <= 1:
        return "today"
    if giorni <= 3:
        return "3days"
    if giorni <= 7:
        return "week"
    return "month"


# --- Adattatori per fonte ---------------------------------------------------

def fetch_adzuna(query, luogo, pagina):
    url = f"https://api.adzuna.com/v1/api/jobs/{PAESE}/search/{pagina}"
    params = {
        "app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY,
        "results_per_page": PER_PAGINA, "what": query,
        "max_days_old": MAX_GIORNI, "content-type": "application/json",
    }
    if luogo:
        params["where"] = luogo
        params["distance"] = DISTANZA_KM
    return [
        riga("Adzuna",
             j.get("title", "").replace("<strong>", "").replace("</strong>", ""),
             (j.get("company") or {}).get("display_name", ""),
             (j.get("location") or {}).get("display_name", ""),
             j.get("description", ""), j.get("redirect_url", ""),
             j.get("created", "")[:10], (j.get("category") or {}).get("label", ""),
             j.get("salary_min"), j.get("salary_max"), query)
        for j in get_json(url, params=params).get("results", [])
    ]


def fetch_jsearch(query, luogo, pagina):
    url = "https://jsearch.p.rapidapi.com/search"
    headers = {"X-RapidAPI-Key": RAPIDAPI_KEY, "X-RapidAPI-Host": "jsearch.p.rapidapi.com"}
    q = f"{query} {luogo}".strip() if luogo else f"{query} Italia"
    params = {"query": q, "page": str(pagina), "num_pages": "1",
              "country": PAESE, "date_posted": giorni_to_jsearch(MAX_GIORNI)}
    risultati = []
    for j in get_json(url, headers=headers, params=params).get("data", []):
        luogo_job = ", ".join(x for x in [j.get("job_city"), j.get("job_country")] if x)
        risultati.append(
            riga("JSearch", j.get("job_title", ""), j.get("employer_name", ""),
                 luogo_job, j.get("job_description", ""), j.get("job_apply_link", ""),
                 (j.get("job_posted_at_datetime_utc") or "")[:10], "",
                 j.get("job_min_salary"), j.get("job_max_salary"), query)
        )
    return risultati


def fetch_jooble(query, luogo, pagina):
    # Jooble vuole la chiave nel path e i parametri nel body: e' il suo contratto API.
    url = f"https://jooble.org/api/{JOOBLE_KEY}"
    risposta = requests.post(url, json={"keywords": query, "location": luogo,
                                        "page": str(pagina)}, timeout=30)
    risposta.raise_for_status()
    return [
        riga("Jooble", j.get("title", ""), j.get("company", ""),
             j.get("location", ""), j.get("snippet", ""), j.get("link", ""),
             (j.get("updated") or "")[:10], j.get("type", ""), None, None, query)
        for j in risposta.json().get("jobs", [])
    ]


def fetch_careerjet(query, luogo, pagina):
    # user_ip e user_agent sono richiesti dal protocollo affiliati; in un contesto
    # server-side non esiste un client reale, quindi si usano placeholder.
    url = "http://public.api.careerjet.net/search"
    params = {"keywords": query, "location": luogo or "Italia", "affid": CAREERJET_AFFID,
              "user_ip": "0.0.0.0", "user_agent": "job-matching-engine/1.0",
              "locale_code": LOCALE, "pagesize": PER_PAGINA, "page": pagina,
              "contenttype": "application/json"}
    return [
        riga("Careerjet", j.get("title", ""), j.get("company", ""),
             j.get("locations", ""), j.get("description", ""), j.get("url", ""),
             (j.get("date") or "")[:10], "", j.get("salary_min"), j.get("salary_max"), query)
        for j in get_json(url, params=params).get("jobs", [])
    ]


def fetch_arbeitnow(pagina):
    # Board europea senza parametro di ricerca: si scarica la pagina e si filtra a valle.
    url = "https://www.arbeitnow.com/api/job-board-api"
    return [
        riga("Arbeitnow", j.get("title", ""), j.get("company_name", ""),
             j.get("location", ""), j.get("description", ""), j.get("url", ""),
             "", ", ".join(j.get("tags", []) or []), None, None, "remote")
        for j in get_json(url, params={"page": pagina}).get("data", [])
    ]


# --- Esecuzione -------------------------------------------------------------
righe = []


def raccogli(nome, funzione, *args):
    """Isola il fallimento di una singola chiamata: la pipeline non si interrompe."""
    try:
        risultati = funzione(*args)
        righe.extend(risultati)
        time.sleep(0.4)                        # cortesia verso le API
        print(f"  [{nome}] +{len(risultati)}")
    except Exception as errore:
        print(f"  ! [{nome}] errore: {errore}")


for query in QUERIES:
    for luogo in LUOGHI:
        if USA_ADZUNA:
            for p in range(1, PAGINE + 1):
                raccogli("Adzuna", fetch_adzuna, query, luogo, p)
        if USA_JSEARCH:
            for p in range(1, PAGINE_ALTRE + 1):
                raccogli("JSearch", fetch_jsearch, query, luogo, p)
        if USA_JOOBLE:
            for p in range(1, PAGINE_ALTRE + 1):
                raccogli("Jooble", fetch_jooble, query, luogo, p)
        if USA_CAREERJET:
            for p in range(1, PAGINE_ALTRE + 1):
                raccogli("Careerjet", fetch_careerjet, query, luogo, p)

if USA_ARBEITNOW:
    grezzi = []
    for p in range(1, PAGINE_ALTRE + 2):
        try:
            grezzi.extend(fetch_arbeitnow(p))
            time.sleep(0.4)
        except Exception as errore:
            print(f"  ! [Arbeitnow] errore: {errore}")
    termini = {w.lower() for q in QUERIES for w in q.split()}
    filtrati = [r for r in grezzi
                if any(t in (r["titolo"] + r["descrizione"]).lower() for t in termini)]
    righe.extend(filtrati)
    print(f"  [Arbeitnow] +{len(filtrati)} (su {len(grezzi)} scaricati, filtrati per keyword)")

df = pd.DataFrame(righe)
print(f"\nTotale grezzo: {len(df)}")
if not df.empty:
    print(df["fonte"].value_counts().to_string())

In [ ]:
# ============================================================================
# CELLA 5 — Deduplica cross-source
# ----------------------------------------------------------------------------
# Lo stesso annuncio compare su piu board con URL diversi (parametri di tracking,
# redirect di affiliazione). Serve una doppia chiave: URL canonico + (titolo, azienda).
# ============================================================================
if df.empty:
    raise SystemExit("Nessun annuncio raccolto: verifica chiavi, quote e parametri di ricerca.")


def normalizza(testo):
    return re.sub(r"\s+", " ", str(testo).lower().strip())


df["link_canonico"] = df["link"].str.split("?").str[0].str.lower().str.rstrip("/")
df["chiave_logica"] = df["titolo"].map(normalizza) + " | " + df["azienda"].map(normalizza)

prima = len(df)
df = df.drop_duplicates(subset=["link_canonico"])
df = df.drop_duplicates(subset=["chiave_logica"])
df = df.drop(columns=["link_canonico", "chiave_logica"]).reset_index(drop=True)

# Campo unico su cui girano embedding e keyword matching
df["testo"] = (df["titolo"].fillna("") + ". " + df["descrizione"].fillna("")).str.strip()

# Registro degli esclusi: ogni filtro annota cosa scarta e perche, cosi le regole
# restano verificabili invece di far sparire silenziosamente degli annunci.
COLONNE_ESCLUSI = ["motivo_esclusione", "titolo", "azienda", "luogo", "fonte", "link"]
df_esclusi = pd.DataFrame(columns=COLONNE_ESCLUSI)


def registra_esclusi(sottoinsieme, motivo):
    global df_esclusi
    if len(sottoinsieme):
        tmp = sottoinsieme[["titolo", "azienda", "luogo", "fonte", "link"]].copy()
        tmp.insert(0, "motivo_esclusione", motivo)
        df_esclusi = pd.concat([df_esclusi, tmp], ignore_index=True)


print(f"Annunci unici: {len(df)} (rimossi {prima - len(df)} duplicati, "
      f"{(prima - len(df)) / prima:.0%} del totale)")

In [ ]:
# ============================================================================
# CELLA 6 — Filtro 1: annunci riservati alle categorie protette
# ----------------------------------------------------------------------------
# Filtro specifico per il mercato italiano: intercetta le offerte legalmente
# riservate a chi rientra nelle liste della L. 68/99. Fuori dall'Italia, o se il
# candidato rientra in quelle categorie, si disattiva da § 5 della configurazione.
#
# Il rischio qui e il falso positivo: quasi ogni annuncio italiano cita la
# normativa in coda per dovere informativo. In modalita "mirata" serve la
# compresenza di un riferimento normativo E di un termine di riserva.
# ============================================================================
TERMINI_NORMATIVI = ["categorie protette", "categoria protetta", "collocamento mirato",
                     "legge 68", "l. 68", "l.68", "68/99"]
TERMINI_RISERVA = ["riservat", "dedicat", "esclusivament", "rivolta esclusivamente",
                   "appartenenza alle categorie", "iscritti alle liste", "iscrizione alle liste"]


def cita_normativa(testo):
    testo = (testo or "").lower()
    return any(t in testo for t in TERMINI_NORMATIVI)


def e_riservato(titolo, descrizione):
    titolo = (titolo or "").lower()
    descrizione = (descrizione or "").lower()
    if cita_normativa(titolo):                       # nel titolo e sempre dirimente
        return True
    if MODALITA_FILTRO == "aggressiva":
        return cita_normativa(descrizione)
    return cita_normativa(descrizione) and any(t in descrizione for t in TERMINI_RISERVA)


if not FILTRO_CATEGORIE_PROTETTE:
    print("Filtro categorie protette: disattivato da configurazione.")
else:
    prima = len(df)
    maschera = df.apply(lambda r: e_riservato(r["titolo"], r["descrizione"]), axis=1)
    registra_esclusi(df[maschera], "Riservato categorie protette")
    df = df[~maschera].reset_index(drop=True)
    print(f"Filtro categorie protette: esclusi {int(maschera.sum())}/{prima}. Rimasti: {len(df)}")

In [ ]:
# ============================================================================
# CELLA 7 — Filtro 2: seniority ed esperienza richiesta
# ----------------------------------------------------------------------------
# Tre segnali in cascata, dal piu affidabile al piu rumoroso:
#   1) marcatori di seniority nel titolo        -> MARCATORI_SENIORITY  (§ 5)
#   2) formule che implicano esperienza lunga   -> FORMULE_PLURIENNALI  (§ 5)
#   3) estrazione numerica degli anni richiesti -> MAX_ANNI_ESPERIENZA  (§ 5)
#
# Il punto 3 e il piu delicato: "5 anni" puo riferirsi all'eta dell'azienda o
# alla durata di un progetto. Si accetta il numero solo se ricade in una finestra
# di contesto attorno a un lemma di "esperienza".
#
# Chi cerca un ruolo senior anziche junior svuota MARCATORI_SENIORITY e porta
# MAX_ANNI_ESPERIENZA a None: il filtro si spegne senza toccare il codice.
# ============================================================================
FINESTRA_CONTESTO = 40   # caratteri attorno al numero entro cui cercare l'ancora

# Pattern per lingua: (espressione regolare, ancore semantiche accettate)
PATTERN_ANNI = [
    (r"(\d+)\s*(?:[-\u2013a]\s*\d+)?\s*\+?\s*ann[io]", ("esperienz", "experience", "seniority")),
    (r"(\d+)\s*(?:[-\u2013to]+\s*\d+)?\s*\+?\s*years?", ("experience", "esperienz")),
]


def anni_richiesti(testo):
    """Massimo numero di anni di esperienza richiesti, None se non determinabile."""
    testo = (testo or "").lower()
    anni = []
    for pattern, ancore in PATTERN_ANNI:
        for match in re.finditer(pattern, testo):
            contesto = testo[max(0, match.start() - FINESTRA_CONTESTO):
                             match.end() + FINESTRA_CONTESTO]
            if any(a in contesto for a in ancore):
                anni.append(int(match.group(1)))
    return max(anni) if anni else None


def e_troppo_senior(titolo, descrizione):
    titolo = (titolo or "").lower()
    descrizione = (descrizione or "").lower()
    for marcatore in MARCATORI_SENIORITY:
        if marcatore in titolo:
            return True, f"titolo: '{marcatore.strip()}'"
    for formula in FORMULE_PLURIENNALI:
        if formula in descrizione:
            return True, "esperienza pluriennale"
    if MAX_ANNI_ESPERIENZA is not None:
        anni = anni_richiesti(descrizione)
        if anni is not None and anni > MAX_ANNI_ESPERIENZA:
            return True, f"{anni} anni richiesti"
    return False, ""


if MAX_ANNI_ESPERIENZA is None and not MARCATORI_SENIORITY and not FORMULE_PLURIENNALI:
    print("Filtro seniority: disattivato da configurazione.")
else:
    prima = len(df)
    valutazione = df.apply(lambda r: e_troppo_senior(r["titolo"], r["descrizione"]), axis=1)
    maschera = valutazione.map(lambda x: x[0])
    motivi = valutazione.map(lambda x: x[1])

    scartati = df[maschera].copy()
    if len(scartati):
        scartati["_motivo"] = motivi[maschera].values
        for motivo, gruppo in scartati.groupby("_motivo"):
            registra_esclusi(gruppo, f"Seniority/esperienza ({motivo})")

    df = df[~maschera].reset_index(drop=True)
    limite = (f"max {MAX_ANNI_ESPERIENZA} anni" if MAX_ANNI_ESPERIENZA is not None
              else "solo marcatori nel titolo")
    print(f"Filtro seniority ({limite}): esclusi {int(maschera.sum())}/{prima}. "
          f"Rimasti: {len(df)}")

In [ ]:
# ============================================================================
# CELLA 8 — Filtro 3: modalita di lavoro e area geografica
# ----------------------------------------------------------------------------
# Le aree e i loro centri sono definiti in § 4 della configurazione: questa cella
# non contiene alcun riferimento a un paese specifico.
#
# Regola asimmetrica: il remoto non ha vincoli geografici, l'on-site e l'ibrido
# si. Gli annunci con sede non determinabile vengono TENUTI e segnalati: scartare
# un lead per un campo mancante e un errore piu costoso che rivederlo a mano.
#
# Tre configurazioni possibili da § 4:
#   AREE_AMMESSE = [...]        filtra on-site/ibrido, ammette il remoto ovunque
#
# La colonna "area" riporta l'area riconosciuta, "Altra area" se identificata
# altrove (un annuncio remoto puo legittimamente trovarsi qui) o vuota se la
# sede non e determinabile.
#   AREE_AMMESSE = []           nessun filtro geografico
#   SOLO_REMOTO  = True         tiene esclusivamente gli annunci da remoto
# ============================================================================
import unicodedata


def senza_accenti(testo):
    """Normalizza per il matching: gli annunci scrivono 'Cuneo' e 'CUNEO', 'Forlì' e 'Forli'."""
    testo = unicodedata.normalize("NFKD", str(testo or "").lower())
    return "".join(c for c in testo if not unicodedata.combining(c))


# Termini che segnalano la modalita di lavoro. Sono multilingue di proposito:
# gli annunci italiani usano spesso l'inglese proprio per questi campi.
TERMINI_REMOTO = ["full remote", "fully remote", "100% remote", "completamente da remoto",
                  "interamente da remoto", "da remoto", "in remoto", "lavoro da remoto",
                  "smart working", "lavoro agile", "remote-first", "remote first",
                  "telelavoro", "remoto", "remote", "homeoffice", "home office",
                  "teletrabajo", "teletravail"]
TERMINI_IBRIDO = ["ibrido", "ibrida", "hybrid", "hibrido", "modalita mista", "lavoro misto"]

# Indice di ricerca costruito dalle aree definite in configurazione
TERMINI_AREA = {area: [senza_accenti(c) for c in citta]
                for area, citta in AREE_GEOGRAFICHE.items()}


def modalita_lavoro(titolo, descrizione, luogo):
    blob = senza_accenti(f"{titolo} {descrizione} {luogo}")
    if any(t in blob for t in TERMINI_IBRIDO):      # l'ibrido prevale: cita anche "remoto"
        return "Ibrido"
    if any(t in blob for t in TERMINI_REMOTO):
        return "Remoto"
    return "In sede"


def area_di(luogo):
    """Area riconosciuta, 'Altra area' se identificata altrove, '' se non determinabile."""
    luogo = senza_accenti(luogo)
    if not luogo.strip():
        return ""
    for area in AREE_AMMESSE:
        if any(t in luogo for t in TERMINI_AREA.get(area, [])):
            return area
    return "Altra area"


df["modalita"] = df.apply(
    lambda r: modalita_lavoro(r["titolo"], r["descrizione"], r["luogo"]), axis=1)
df["area"] = df["luogo"].map(area_di) if AREE_AMMESSE else ""

prima = len(df)
if SOLO_REMOTO:
    maschera = df["modalita"] != "Remoto"
    motivo = "Non remoto (ricerca solo remota)"
elif AREE_AMMESSE:
    # Si esclude solo cio che e certamente fuori area e non remoto.
    maschera = (df["modalita"] != "Remoto") & (df["area"] == "Altra area")
    motivo = f"Fuori area ({'/'.join(AREE_AMMESSE)}), non remoto"
else:
    maschera = pd.Series(False, index=df.index)
    motivo = ""

if motivo:
    registra_esclusi(df[maschera], motivo)
    df = df[~maschera].reset_index(drop=True)
    print(f"Filtro geografico: esclusi {int(maschera.sum())}/{prima}. Rimasti: {len(df)}")
else:
    print(f"Filtro geografico: disattivato. Annunci: {len(df)}")

print(f"  modalita: {df['modalita'].value_counts().to_dict()}")
if AREE_AMMESSE and not SOLO_REMOTO:
    da_verificare = int(((df["modalita"] != "Remoto") & (df["area"] == "")).sum())
    if da_verificare:
        print(f"  {da_verificare} annunci con sede non determinata: tenuti, da verificare a mano")

In [ ]:
# ============================================================================
# CELLA 9 — Scoring ibrido: embedding semantici + keyword
# ----------------------------------------------------------------------------
# Il modello multilingue proietta italiano e inglese nello stesso spazio, quindi
# "analisi delle serie storiche" e "time series forecasting" finiscono vicini.
# La similarita coseno grezza su testi lunghi si concentra in una banda stretta
# (~0.10-0.55): si riscala linearmente sulla banda SIM_MIN/SIM_MAX (§ 6) per
# ottenere un punteggio 0-100 leggibile invece di una classifica schiacciata.
# ============================================================================
import numpy as np
from sentence_transformers import SentenceTransformer, util

if df.empty:
    raise SystemExit("Nessun annuncio sopravvissuto ai filtri: allarga query, luoghi o regioni.")

print("Caricamento del modello di embedding (al primo avvio ~1 min)...")
modello_embedding = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

emb_profilo = modello_embedding.encode(
    PROFILO_CANDIDATO, convert_to_tensor=True, normalize_embeddings=True)
emb_annunci = modello_embedding.encode(
    df["testo"].tolist(), convert_to_tensor=True, normalize_embeddings=True,
    show_progress_bar=True, batch_size=32)

similarita = util.cos_sim(emb_profilo, emb_annunci).cpu().numpy().flatten()
punteggio_semantico = np.clip((similarita - SIM_MIN) / (SIM_MAX - SIM_MIN), 0, 1) * 100


def punteggio_keyword(testo, saturazione=SATURAZIONE_KEYWORD):
    """Conta le skill presenti, con saturazione: oltre N match il segnale non cresce."""
    testo = (testo or "").lower()
    trovate = sum(1 for skill in SKILL_CHIAVE if skill.strip() in testo)
    return min(trovate / saturazione, 1.0) * 100


punteggio_kw = df["testo"].apply(punteggio_keyword).to_numpy()

df["affinita"] = (PESO_SEMANTICO * punteggio_semantico + PESO_KEYWORD * punteggio_kw).round(1)
df["score_semantico"] = punteggio_semantico.round(1)
df["score_keyword"] = punteggio_kw.round(1)

print(f"Scoring completato su {len(df)} annunci. "
      f"Affinita media: {df['affinita'].mean():.1f} | max: {df['affinita'].max():.1f}")

In [ ]:
# ============================================================================
# CELLA 10 — Anteprima della classifica
# ============================================================================
pd.set_option("display.max_colwidth", 60)

df = df.sort_values("affinita", ascending=False).reset_index(drop=True)
colonne = ["affinita", "score_semantico", "score_keyword", "titolo", "azienda",
           "luogo", "area", "modalita", "pubblicato", "fonte"]

print(f"TOP 20 su {len(df)} annunci\n")
display(df[[c for c in colonne if c in df.columns]].head(20))

In [ ]:
# ============================================================================
# CELLA 11 — Re-ranking LLM sui migliori candidati
# ----------------------------------------------------------------------------
# Lo scoring semantico misura la somiglianza fra due testi, non l'idoneita a una
# candidatura: non sa distinguere un annuncio "vicino ma irraggiungibile" da uno
# realmente accessibile. L'LLM fa questa valutazione, ma solo sui top-N: applicarlo
# a tutti gli annunci sarebbe costoso e superfluo.
# Output vincolato a JSON per rendere la risposta processabile a valle.
# ============================================================================
import json
import random

if not USA_LLM:
    print("ANTHROPIC_API_KEY non configurata: re-ranking LLM saltato.")
    for colonna in ["punteggio_llm", "motivazione_llm", "gap_llm"]:
        df[colonna] = ""
else:
    import anthropic

    # Verifica il nome del modello corrente su docs.claude.com/en/docs/about-claude/models
    MODELLO_LLM = "claude-sonnet-4-5"
    TOP_N = 30            # quanti annunci valutare
    MAX_TENTATIVI = 5     # retry sui codici transitori

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    df = df.sort_values("affinita", ascending=False).reset_index(drop=True)

    CODICI_TRANSITORI = {429, 500, 502, 503, 529}

    def chiama_con_backoff(prompt):
        """Backoff esponenziale con jitter: i 529 sono frequenti nelle ore di punta."""
        for tentativo in range(1, MAX_TENTATIVI + 1):
            try:
                risposta = client.messages.create(
                    model=MODELLO_LLM, max_tokens=300,
                    messages=[{"role": "user", "content": prompt}])
                return risposta.content[0].text
            except (anthropic.APIStatusError, anthropic.APIConnectionError,
                    anthropic.RateLimitError) as errore:
                codice = getattr(errore, "status_code", None)
                transitorio = (codice in CODICI_TRANSITORI
                               or isinstance(errore, anthropic.APIConnectionError))
                if not transitorio or tentativo == MAX_TENTATIVI:
                    raise
                attesa = min(2 ** tentativo, 30) + random.uniform(0, 1)
                print(f"    {codice}: nuovo tentativo fra {attesa:.1f}s "
                      f"[{tentativo}/{MAX_TENTATIVI}]")
                time.sleep(attesa)

    def valuta_annuncio(titolo, azienda, descrizione):
        prompt = f"""Sei un recruiter tecnico. Valuta l'affinita fra questo candidato e l'annuncio.
Il candidato cerca un primo ruolo junior (0-1 anni di esperienza).

CANDIDATO:
{PROFILO_CANDIDATO}

ANNUNCIO:
Titolo: {titolo}
Azienda: {azienda}
Descrizione: {descrizione[:2500]}

Rispondi SOLO con un JSON valido, senza testo aggiuntivo:
{{"punteggio": <0-100>, "motivazione": "<max 20 parole>", "gap_principale": "<max 12 parole>"}}"""
        try:
            testo = chiama_con_backoff(prompt)
        except Exception as errore:
            return {"punteggio": None, "motivazione": f"errore: {errore}", "gap_principale": ""}

        testo = (testo or "").strip().replace("```json", "").replace("```", "").strip()
        try:
            return json.loads(testo)
        except json.JSONDecodeError:
            return {"punteggio": None, "motivazione": testo[:80], "gap_principale": ""}

    for colonna in ["punteggio_llm", "motivazione_llm", "gap_llm"]:
        if colonna not in df.columns:
            df[colonna] = ""

    n_valutati = min(TOP_N, len(df))
    for posizione in range(n_valutati):
        annuncio = df.loc[posizione]
        esito = valuta_annuncio(annuncio["titolo"], annuncio["azienda"], annuncio["descrizione"])
        df.at[posizione, "punteggio_llm"] = esito.get("punteggio")
        df.at[posizione, "motivazione_llm"] = esito.get("motivazione")
        df.at[posizione, "gap_llm"] = esito.get("gap_principale")
        print(f"  {posizione + 1}/{n_valutati}  {str(annuncio['titolo'])[:50]}")
        time.sleep(0.3)

    validi = pd.to_numeric(df["punteggio_llm"], errors="coerce").dropna()
    print(f"\nRe-ranking completato su {n_valutati} annunci. "
          f"Punteggio LLM medio: {validi.mean():.1f}" if len(validi) else
          f"\nRe-ranking completato su {n_valutati} annunci.")

In [ ]:
# ============================================================================
# CELLA 12 — Export
# ----------------------------------------------------------------------------
# Tre output: Annunci (risultati della run), Tracker (candidature, persistente e
# incrementale) ed Esclusi (audit dei filtri).
# Su Colab scrive su Google Sheets; in locale, o se l'autenticazione fallisce,
# ripiega su CSV senza interrompere la pipeline.
# ============================================================================
from pathlib import Path

NOME_FOGLIO = "Job Matching Engine"    # foglio Google, creato se non esiste
SOGLIA_TRACKER = 60                    # affinita minima per entrare nel Tracker
INDICI_TRACKER = []                    # in alternativa, indici espliciti es. [0, 1, 4]
CARTELLA_OUTPUT = Path("output")

COLONNE_ANNUNCI = ["affinita", "punteggio_llm", "motivazione_llm", "gap_llm", "titolo",
                   "azienda", "luogo", "area", "modalita", "categoria", "pubblicato",
                   "salario_min", "salario_max", "fonte", "query", "link", "descrizione"]
COLONNE_TRACKER = ["data_candidatura", "azienda", "ruolo", "modalita", "area", "piattaforma",
                   "link_annuncio", "affinita", "punteggio_llm", "versione_cv", "lettera",
                   "referente", "stato", "data_risposta", "tipo_risposta",
                   "prossimo_followup", "note"]

classifica = df.sort_values("affinita", ascending=False).reset_index(drop=True)
annunci = classifica[[c for c in COLONNE_ANNUNCI if c in classifica.columns]].fillna("")

selezionati = (classifica.loc[INDICI_TRACKER] if INDICI_TRACKER
               else classifica[classifica["affinita"] >= SOGLIA_TRACKER])

nuove_candidature = pd.DataFrame({
    "data_candidatura": "",
    "azienda": selezionati["azienda"].values,
    "ruolo": selezionati["titolo"].values,
    "modalita": selezionati["modalita"].values,
    "area": selezionati["area"].values,
    "piattaforma": selezionati["fonte"].values,
    "link_annuncio": selezionati["link"].values,
    "affinita": selezionati["affinita"].values,
    "punteggio_llm": selezionati["punteggio_llm"].values,
    "versione_cv": "", "lettera": "", "referente": "", "stato": "Da candidare",
    "data_risposta": "", "tipo_risposta": "", "prossimo_followup": "", "note": "",
}, columns=COLONNE_TRACKER)

esclusi = (df_esclusi.drop_duplicates(subset=["link"]).reset_index(drop=True)
           if len(df_esclusi) else pd.DataFrame(columns=COLONNE_ESCLUSI))

# --- Tentativo su Google Sheets ---------------------------------------------
scritto_su_sheets = False
if IN_COLAB:
    try:
        import gspread
        from google.auth import default
        from google.colab import auth
        from gspread_dataframe import get_as_dataframe, set_with_dataframe

        auth.authenticate_user()
        credenziali, _ = default()
        gc = gspread.authorize(credenziali)

        try:
            foglio = gc.open(NOME_FOGLIO)
        except gspread.SpreadsheetNotFound:
            foglio = gc.create(NOME_FOGLIO)

        def scrivi_scheda(nome, dati, righe_extra=10):
            try:
                ws = foglio.worksheet(nome)
                ws.clear()
            except gspread.WorksheetNotFound:
                ws = foglio.add_worksheet(nome, rows=max(len(dati) + righe_extra, 100),
                                          cols=max(len(dati.columns) + 2, 10))
            set_with_dataframe(ws, dati)
            return ws

        scrivi_scheda("Annunci", annunci)
        print(f"Scheda 'Annunci': {len(annunci)} righe.")

        # Il Tracker e' persistente: si accoda senza reintrodurre annunci gia presenti.
        try:
            ws_tracker = foglio.worksheet("Tracker")
            tracker = (get_as_dataframe(ws_tracker).dropna(how="all")
                       .reindex(columns=COLONNE_TRACKER))
        except gspread.WorksheetNotFound:
            ws_tracker = foglio.add_worksheet("Tracker", rows=400,
                                              cols=len(COLONNE_TRACKER) + 2)
            tracker = pd.DataFrame(columns=COLONNE_TRACKER)

        gia_presenti = set(tracker["link_annuncio"].dropna()) if "link_annuncio" in tracker else set()
        da_aggiungere = nuove_candidature[~nuove_candidature["link_annuncio"].isin(gia_presenti)]
        tracker = pd.concat([tracker, da_aggiungere], ignore_index=True).fillna("")
        ws_tracker.clear()
        set_with_dataframe(ws_tracker, tracker)
        print(f"Scheda 'Tracker': +{len(da_aggiungere)} nuove, {len(tracker)} totali.")

        scrivi_scheda("Esclusi", esclusi.fillna(""))
        print(f"Scheda 'Esclusi': {len(esclusi)} righe.")
        print(f"\nFoglio: {foglio.url}")
        scritto_su_sheets = True

    except Exception as errore:
        print(f"Google Sheets non disponibile ({errore}); ripiego su CSV.")

# --- Fallback su CSV ---------------------------------------------------------
if not scritto_su_sheets:
    CARTELLA_OUTPUT.mkdir(exist_ok=True)
    oggi = date.today().isoformat()
    annunci.to_csv(CARTELLA_OUTPUT / f"annunci_{oggi}.csv", index=False)
    nuove_candidature.to_csv(CARTELLA_OUTPUT / f"tracker_{oggi}.csv", index=False)
    esclusi.to_csv(CARTELLA_OUTPUT / f"esclusi_{oggi}.csv", index=False)
    print(f"Esportati in {CARTELLA_OUTPUT.resolve()}/ "
          f"({len(annunci)} annunci, {len(nuove_candidature)} candidature, {len(esclusi)} esclusi)")

# --- Riepilogo ---------------------------------------------------------------
print("\n--- Riepilogo run ---")
print(f"Blocco eseguito      : {BLOCCO_OGGI}")
print(f"Annunci in classifica: {len(annunci)}")
print(f"Sopra soglia ({SOGLIA_TRACKER})   : {len(selezionati)}")
if len(esclusi):
    print("Esclusioni per motivo:")
    print(esclusi["motivo_esclusione"].value_counts().to_string())

## Note operative

**Ordine di esecuzione:** le celle vanno eseguite in sequenza, dalla 1 alla 12. Ogni cella dipende dallo stato lasciato dalla precedente.

**Costo per run:** con Adzuna e Arbeitnow attive e il re-ranking LLM sui primi 30 annunci, una run costa qualche centesimo di euro e circa due minuti, di cui la maggior parte è il caricamento del modello di embedding (solo alla prima esecuzione della sessione).

---

## Problemi noti e come si manifestano

| Sintomo | Causa | Rimedio |
|---|---|---|
| `[JSearch] errore: 404` | Quota RapidAPI esaurita o sottoscrizione non attiva | Verifica il piano su RapidAPI; la pipeline prosegue senza quella fonte |
| Pochissimi annunci dopo i filtri | `AREE_AMMESSE` troppo stretta, o `MAX_GIORNI` basso | Allarga le aree o alza `MAX_GIORNI` (§ 4 e § 6) |
| Molti falsi positivi su "Remoto" | `TERMINI_REMOTO` contiene voci generiche (`remote`, `remoto`) che matchano anche frasi negative del tipo *"no remote work"* | Restringere la lista ai soli termini espliciti, o aggiungere un controllo sulla negazione |
| `AssertionError` all'avvio | Un'area in `AREE_AMMESSE` non è definita in `AREE_GEOGRAFICHE`, o i pesi non sommano a 1 | Il messaggio indica il paragrafo della configurazione da correggere |
| Città della tua zona non riconosciuta | Non è nell'elenco dell'area | Aggiungila alla lista in `AREE_GEOGRAFICHE` (§ 4): il matching è per sottostringa |
| `punteggio_llm` vuoto su alcune righe | Il modello ha risposto fuori dallo schema JSON | La motivazione contiene i primi 80 caratteri della risposta grezza, utile per capire cosa è andato storto |
| Affinità tutte simili | La banda `SIM_MIN`/`SIM_MAX` non è tarata su questo profilo | Ricalibra i due estremi osservando la distribuzione di `score_semantico` |

## Limiti dell'approccio

- **La deduplica su (titolo, azienda) è aggressiva.** Due posizioni realmente distinte con lo stesso titolo nella stessa azienda vengono collassate in una. Nel contesto d'uso è un compromesso accettabile, in un sistema di produzione no.
- **Il filtro sugli anni di esperienza è euristico.** La finestra di 40 caratteri attorno al numero riduce i falsi positivi ma non li elimina: formulazioni come *"il progetto dura 3 anni, cerchiamo neolaureati"* possono ancora ingannarlo.
- **La banda di riscalatura è calibrata empiricamente** sul profilo di esempio. Cambiando profilo va ritarata osservando la distribuzione di `score_semantico`: è il parametro che più incide sulla leggibilità della classifica.
- **Il riconoscimento delle aree si basa su liste di città compilate a mano.** Copre i centri principali, non i comuni minori: un annuncio in un paese non elencato finisce fra i "non determinati" e va rivisto a mano. Una geocodifica vera risolverebbe il problema al costo di un'altra dipendenza esterna.
- **L'LLM valuta un annuncio alla volta**, senza contesto comparativo: i punteggi sono coerenti in media ma non strettamente confrontabili fra loro.

## Sviluppi possibili

- Sostituire la deduplica esatta con un match fuzzy sui titoli (Levenshtein normalizzata) per intercettare le varianti minime.
- Valutare gli annunci a batch, così che il modello possa ordinarli l'uno rispetto all'altro invece che in isolamento.
- Persistere lo storico in SQLite per tracciare quali query rendono di più nel tempo e potare quelle improduttive.
- Estrarre le skill richieste dagli annunci ad alta affinità per identificare le lacune ricorrenti del profilo.

---

## Riusare la pipeline

Per adattarla alla propria ricerca serve modificare **solo la cella di configurazione**. In ordine di importanza:

1. `PROFILO_CANDIDATO` (§ 1) — è ciò su cui si basa l'intero ranking.
2. `BLOCCHI` (§ 2) — i ruoli da cercare, in numero libero.
3. `AREE_GEOGRAFICHE` e `AREE_AMMESSE` (§ 4) — le zone di interesse con i rispettivi centri.
4. `SKILL_CHIAVE` (§ 3) — le tecnologie del proprio settore.
5. `MAX_ANNI_ESPERIENZA` e `MARCATORI_SENIORITY` (§ 5) — da invertire o svuotare se si cerca un ruolo senior.
6. `FILTRO_CATEGORIE_PROTETTE` (§ 5) — da disattivare fuori dall'Italia.

Dopo la prima run con un profilo nuovo, controlla `score_semantico` e ritara `SIM_MIN`/`SIM_MAX` (§ 6) se i punteggi risultano schiacciati.

---

*Progetto personale. Le API sono usate nel rispetto dei rispettivi termini di servizio; nessun dato degli annunci viene ridistribuito in questo repository.*